# Nova — Smart Home Assistant

One-click entry point. Run cells top-to-bottom.


In [7]:
# Cell 1: Install dependencies (first run only)
import sys
!{sys.executable} -m pip install -q numpy pandas sounddevice soundfile faster-whisper \
    transformers accelerate sentencepiece pyttsx3 chromadb sentence-transformers bitsandbytes


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3 install --upgrade pip


In [8]:
# Cell 2: Add project root to sys.path + HF authentication
import sys, os

PROJECT_ROOT = os.path.dirname(os.path.abspath("main.ipynb"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

# Set HF_TOKEN in terminal before launching Jupyter:
#   export HF_TOKEN=hf_...
from huggingface_hub import login
token = os.environ.get("HF_TOKEN", "")
if token:
    login(token=token)
    print("HF login done.")
else:
    print("HF_TOKEN not set — public models still work.")


Project root: /Users/yiwenchen/Desktop/CSEE6895/final_project/6895_Final_project
HF_TOKEN not set — public models still work.


In [9]:
# Cell 3: Load LLM parser
from llm_parser import LLMParser

llm = LLMParser()

Loading LLM (Qwen/Qwen2.5-3B-Instruct) on mps [torch.float16] ...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM ready.


In [10]:
# Cell 4: Load STT + TTS
from audio import STTModel, TTSEngine

stt = STTModel()
tts = TTSEngine()

Loading STT (tiny.en) ...
STT ready.


In [11]:
# Cell 5: Load embedding model + initialize memory
from sentence_transformers import SentenceTransformer
from memory import MemoryManager

_embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embed = lambda text: _embed_model.encode(text, convert_to_numpy=True).tolist()

memory = MemoryManager(embed_fn=embed)
print("Memory summary:", memory.summary())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Memory summary: {'working_turns': 0, 'episodic_count': 7, 'prefs': {}, 'skills_count': 0}


In [12]:
# Cell 6: Create Nova agent
from agent import NovaAgent

nova = NovaAgent(llm=llm, memory=memory, speak=tts.speak)
print('Nova agent ready.')


Nova agent ready.


In [13]:
# Cell 7: Text demo

SEP = '=' * 62

def run_test(label, text, agent, reset=True):
    if reset:
        agent.reset_dialogue()
    print('\n' + SEP)
    print('[' + label + ']')
    print('  input : ' + repr(text))
    result = agent.handle(text, verbose=True)
    t     = result['semantic'].get('type', '?')
    reply = result.get('spoken_reply', '-')
    ms    = result.get('llm_latency_ms', '?')
    exec_ = result.get('execution', '-')
    print('  type  : ' + str(t))
    print('  reply : ' + repr(reply))
    print('  exec  : ' + str(exec_) + '  latency: ' + str(ms) + ' ms')
    return result

run_test('direct_command',    'Nova, turn on the light.', nova)
run_test('needs_clarification', 'Nova, I feel a bit cold.', nova)
run_test('no_wakeword',       'How are you today?', nova)

print('\n' + SEP)
print('[memory_test] tell name then recall')
nova.reset_dialogue()
run_test('tell_name',   'Nova, my name is Alice.', nova, reset=False)
run_test('recall_name', 'Nova, what is my name?',  nova, reset=False)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[direct_command]
  input : 'Nova, turn on the light.'
STT text: Nova, turn on the light.
Raw output: {"type":"direct_command","device":"light","action":"turn_on","value":null,"reply":"Sure, turning on the light!"}
Latency: 8682.6 ms
[TTS] Sure, turning on the light!
  type  : direct_command
  reply : 'Sure, turning on the light!'
  exec  : LIGHT -> ON  latency: 8682.64 ms

[needs_clarification]
  input : 'Nova, I feel a bit cold.'
STT text: Nova, I feel a bit cold.
Raw output: {"type":"needs_clarification","question":"Would you like me to close the window or increase the AC temperature?","options":["close_window","increase_ac_temperature"],"reply":"Would you like me to close the window or increase the AC temperature?"}
Latency: 8654.7 ms
[Clarification options]
  [1] Close the window
  [2] Increase ac temperature
[TTS] Would you like me to close the window or increase the AC temperature? Here are your options: Option 1: Close the window. Option 2: Increase ac temperature. Please say t

{'prefilter_passed': True,
 'semantic': {'type': 'general_qa', 'answer': 'Your name is Alice.'},
 'valid': True,
 'reason': 'general_qa_answered',
 'execution': 'NO_DEVICE_ACTION',
 'spoken_reply': 'Your name is Alice.',
 'llm_latency_ms': 5003.812}

In [ ]:
# Cell 9: Start continuous audio loop (microphone required)
# Uncomment to run; press Ctrl+C to stop.

# from audio import AudioListener
# listener = AudioListener(agent=nova, stt=stt)
# listener.continuous_loop()